# 01. 원본 분석 재현

팀 프로젝트 노트북 `4조_팀프로젝트_코드.ipynb` 의 분석 로직을 **그대로** 실행한다.

변경한 것은 다음 두 가지뿐이며, 분석 논리는 손대지 않았다.

- Google Drive 마운트 경로 → 로컬 CSV 경로
- 그래프 저장 코드 추가 (원본은 `plt.show()` 만 수행)

버그로 보이는 부분도 **수정하지 않고 그대로 재현**한다. 수정본은 `02_improved_verification.ipynb` 에서 별도로 다룬다.


In [ ]:
# ── 원본 데이터 경로 ──────────────────────────────────────────────────
# 아래 한 줄만 본인 환경에 맞게 수정하면 됩니다.
# 또는 셸에서 export ER_CAERS_CSV=/path/to/CAERS_ASCII_2004_2017Q2.csv
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

os.environ.setdefault("ER_CAERS_CSV", "data/CAERS_ASCII_2004_2017Q2.csv")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib          # 한글 폰트
except ImportError:
    print("[안내] pip install -r requirements.txt 를 실행하세요")
plt.rcParams.update({"figure.dpi":110, "savefig.dpi":130, "figure.facecolor":"white",
                     "savefig.facecolor":"white", "axes.unicode_minus":False})

# ── 저장 위치: 실행 디렉터리와 무관하게 프로젝트 루트를 판별 ──────────
def find_project_root(start=None):
    """requirements.txt 가 있는 디렉터리를 프로젝트 루트로 본다."""
    cur = Path(start or os.getcwd()).resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "requirements.txt").exists() and (cand / "notebooks").is_dir():
            return cand
    return cur.parent if cur.name == "notebooks" else cur

PROJECT_ROOT = find_project_root()
FIG_DIR = PROJECT_ROOT / "figures"
RES_DIR = PROJECT_ROOT / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# 상대경로면 프로젝트 루트 기준으로 해석
_p = Path(os.environ["ER_CAERS_CSV"])
CSV_PATH = _p if _p.is_absolute() else (PROJECT_ROOT / _p)
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"원본 CSV를 찾을 수 없습니다: {CSV_PATH} | "
        "data/README.md 의 안내에 따라 CAERS 원본을 data/ 에 두거나, "
        "os.environ['ER_CAERS_CSV'] 에 실제 경로를 지정하세요.")

print("CSV       :", CSV_PATH)
print("루트      :", PROJECT_ROOT)
print("저장 위치 :", RES_DIR, "|", FIG_DIR)

## 1. 로드와 컬럼명 변경 (원본 cell 5~6)

In [ ]:
df = pd.read_csv(CSV_PATH)
df.rename(columns={"PRI_Reported Brand/Product Name":"products_name",
                   "SYM_One Row Coded Symptoms":"symptoms",
                   "CI_Gender":"gender",
                   "CI_Age at Adverse Event":"age",
                   "CI_Age Unit":"age_unit",
                   "RA_Report #":"ra_report",
                   "RA_CAERS Created Date":"created_date",
                   "AEC_Event Start Date":"start_date",
                   "PRI_Product Role":"products_role",
                   "PRI_FDA Industry Code":"industry_code",
                   "AEC_One Row Outcomes":"outcomes",
                   "PRI_FDA Industry Name":"products_types"}, inplace=True)

print(f"행 수: {len(df):,}")
print(f"컬럼 수: {len(df.columns)}")
print(f"고유 ra_report: {df.ra_report.nunique():,}")

## 2. 연령 단위 통일 (원본 cell 7)

원본 코드를 그대로 사용한다. 다음 특성이 그대로 유지된다.

- `Day(s)` 는 값과 무관하게 **0세**로 처리
- `Week(s)` / `Month(s)` 는 `int()` 로 **절삭**, `Year(s)` 는 절삭하지 않음
- 120세 초과는 `None`

In [ ]:
def convert_age(x):
    age = x['age']
    unit = x['age_unit']

    if unit == 'Day(s)':
        result = 0
    elif unit == 'Week(s)':
        result = int(age / 52)
    elif unit == 'Month(s)':
        result = int(age / 12)
    elif unit == 'Year(s)':
        result = age
    elif unit == 'Decade(s)':
        result = age * 10
    else:
        return None
    if result > 120:
        return None
    return result

df['Age_Year'] = df.apply(convert_age, axis=1)
print(f"Age_Year 유효: {df.Age_Year.notna().sum():,} / 결측: {df.Age_Year.isna().sum():,}")
print("\nage_unit 분포:")
print(df.age_unit.value_counts(dropna=False))

## 3. 날짜 변환과 연령대 (원본 cell 8, 10)

In [ ]:
df['start_date']   = pd.to_datetime(df['start_date'],   format='%m/%d/%Y')
df['created_date'] = pd.to_datetime(df['created_date'], format='%m/%d/%Y')

def 연령대(age):
    if age is None: return None
    if age <= 5 : return '영유아(0-5)'
    if age <= 19 : return '청소년(6-19)'
    if age <= 59 : return '성인(20-59)'
    return '노인(60+)'

df1 = df.copy()
df1 = df1.dropna(subset=['Age_Year'])
df['age_cate'] = df1['Age_Year'].apply(연령대)

counts = df['age_cate'].value_counts()
expect = {'영유아(0-5)':2964, '청소년(6-19)':3007, '성인(20-59)':26045, '노인(60+)':20890}
print(f"{'연령대':14s} {'재현':>8} {'보고서':>8}  일치")
for k, v in expect.items():
    got = counts.get(k, 0)
    print(f"{k:14s} {got:>8,} {v:>8,}  {'O' if got == v else 'X'}")
print(f"{'합계':14s} {counts.sum():>8,} {sum(expect.values()):>8,}")

## 4. 심각 부작용 정의 (원본 cell 11)

`DEATH`, `HOSPITALIZATION`, `LIFE THREATENING` 중 하나라도 포함되면 심각으로 판정한다.

In [ ]:
def serious(outcome):
    if pd.isna(outcome):
        return False
    return any(k in outcome for k in ['DEATH', 'HOSPITALIZATION', 'LIFE THREATENING'])

df['serious'] = df['outcomes'].apply(serious)
print(f"serious 행: {df.serious.sum():,} / {len(df):,} = {df.serious.mean()*100:.2f}%")

### outcome 항목 실측

`AEC_One Row Outcomes` 에 실제로 등장하는 값을 확인한다.

In [ ]:
from collections import Counter
c = Counter()
for v in df['outcomes'].dropna():
    for t in v.split(','):
        c[t.strip()] += 1

print(f"{'outcome 항목':45s} {'행 수':>8} {'비율':>7}")
for k, v in c.most_common():
    print(f"{k:45s} {v:>8,} {v/len(df)*100:6.2f}%")

## 5. 연도별 신고 건수 추이 (원본 cell 12)

In [ ]:
df['year'] = df['created_date'].dt.year
year_total   = df.groupby('year').size()
year_serious = df[df['serious']].groupby('year').size()
year_total   = year_total[(year_total.index >= 2004) & (year_total.index <= 2017)]
year_serious = year_serious[(year_serious.index >= 2004) & (year_serious.index <= 2017)]

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(year_total.index, year_total.values, color='#185FA5', marker='o', linewidth=2, label='전체 신고')
ax.plot(year_serious.index, year_serious.values, color='#E24B4A', marker='o', linewidth=2, label='심각 신고')
for x, y in zip(year_total.index, year_total.values):
    ax.text(x, y + 50, f'{y:,}', ha='center', color='#185FA5', fontsize=8)
for x, y in zip(year_serious.index, year_serious.values):
    ax.text(x, y + 50, f'{y:,}', ha='center', color='#E24B4A', fontsize=8)
ax.set_xlabel('연도'); ax.set_ylabel('신고 건수 (행 기준)')
ax.set_title('연도별 신고 건수 추이 [원본 재현]', fontweight='bold')
ax.legend(); ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_xticks(year_total.index)
plt.tight_layout()
plt.savefig(FIG_DIR / 'orig_01_yearly_trend.png', bbox_inches='tight')
plt.show()

print("2017년은 Q2까지만 집계된 부분 연도입니다.")

## 6. 산업군별 심각 부작용 비율 (원본 cell 14)

표본 100건 이상, 심각 비율 상위 10개.

In [ ]:
total = df.groupby('products_types').size()
severity = df[df['serious']].groupby('products_types').size()
rate = (severity / total * 100).fillna(0)
top10 = rate[total >= 100].sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5)); ax2 = ax.twinx()
ax.bar(range(len(top10)), top10.values, color='#E24B4A', alpha=0.85, width=0.4, label='심각 비율 (%)')
ax2.bar([x+0.4 for x in range(len(top10))], total[top10.index].values,
        color='#B5D4F4', alpha=0.7, width=0.4, label='전체 건수')
ax.set_xticks([x+0.2 for x in range(len(top10))])
ax.set_xticklabels([l[:20] for l in top10.index], rotation=40, ha='right')
ax.set_ylabel('심각한 부작용 비율(%)', color='#E24B4A')
ax2.set_ylabel('전체 신고 건수 (행)', color='#185FA5')
ax.set_title('산업군별 심각 부작용 비율 [원본 재현]', fontweight='bold', fontsize=13)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
for i, v in enumerate(top10.values):
    ax.text(i, v+0.3, f'{v:.1f}%', ha='center', color='#E24B4A', fontweight='bold', fontsize=9)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'orig_02_industry_serious.png', bbox_inches='tight')
plt.show()

print("보고서 주장 대조")
for k, claim in [('Dietary Conv Food/Meal Replacements', 45.1),
                 ('Vit/Min/Prot/Unconv Diet(Human/Animal)', 34.8),
                 ('Cosmetics', 14.0)]:
    print(f"  {k[:42]:42s} 재현 {rate[k]:5.2f}%  주장 {claim}%")

## 7. 브랜드별 신고 건수 vs 심각 비율 (원본 cell 17)

조건: 20행 이상 → REDACTED 제외 → 신고량 상위 80개.

In [ ]:
brand_df = (df.groupby('products_name')
              .agg(total=('serious','size'), serious=('serious','sum'),
                   industry=('products_types', lambda x: x.mode()[0]))
              .query('total >= 20')
              .loc[lambda x: x.index != 'REDACTED']
              .nlargest(80, 'total'))
brand_df['ratio'] = brand_df['serious'] / brand_df['total'] * 100

fig, ax = plt.subplots(figsize=(11, 7))
ax.axvline(brand_df['total'].median(), color='gray', linestyle='--', alpha=0.4)
ax.axhline(brand_df['ratio'].median(), color='gray', linestyle='--', alpha=0.4)
for brand, row in brand_df.iterrows():
    ax.scatter(row['total'], row['ratio'], s=max(30, np.sqrt(row['total'])*3),
               color='#185FA5', alpha=0.55, edgecolors='white', linewidths=0.5)

주요브랜드 = ['HERBALIFE CELL ACTIVATOR', 'PLEXUS SLIM', 'OXYELITE PRO',
            'ALL DAY ENERGY GREENS', 'RAW OYSTERS', 'HYDROXYCUT']
hl = brand_df[brand_df.index.isin(주요브랜드)]
ax.scatter(hl['total'], hl['ratio'], s=140, color='#E24B4A',
           edgecolors='white', linewidths=1.0, zorder=5)
for brand, row in hl.iterrows():
    ax.annotate(brand[:22], (row['total'], row['ratio']),
                textcoords='offset points', xytext=(6, 4), fontsize=9, fontweight='bold')

ax.set_xscale('log')
ax.set_xlabel('총 신고 건수 (행 기준, 로그 스케일)')
ax.set_ylabel('심각 부작용 비율 (%)')
ax.set_title('브랜드별 신고 건수 vs 심각 부작용 비율 [원본 재현]', fontweight='bold', fontsize=13)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIG_DIR / 'orig_03_brand_scatter.png', bbox_inches='tight')
plt.show()

print(f"대상 브랜드 {len(brand_df)}종 · 최소 {brand_df.total.min()}행")
print("\n빨간 점(라벨 표시 대상) — 코드에 하드코딩된 목록")
print(hl[['total','serious','ratio']].round(2).to_string())

### 이상치 탐지 로직 확인

보고서는 "통계적 이상치 브랜드 발견"이라고 기술한다. 코드에 해당 로직이 있는지 확인한다.

In [ ]:
print("원본 노트북 cell 16~17에서 브랜드 선별에 사용된 연산")
print("  1) query('total >= 20')        — 표본 하한")
print("  2) nlargest(80, 'total')       — 신고량 상위 80개")
print("  3) axvline/axhline(median)     — 중앙값 보조선 (시각 요소)")
print("  4) 주요브랜드 = [...]           — 라벨 표시용 하드코딩 목록 6~7종")
print()
print("→ IQR, z-score, Isolation Forest 등 이상치 탐지 알고리즘은 존재하지 않는다.")
print("→ 중앙값 보조선은 그려지기만 할 뿐, 어떤 필터에도 사용되지 않는다.")
print("→ 실제 선별은 산점도 우상단 영역을 육안으로 판별하는 방식이었다. (수행자 확인)")

## 8. 부작용 분포 도넛 (원본 cell 18)

보고서의 **43.9%** 가 산출된 그래프다.

In [ ]:
data_counts = pd.Series({
    '사망':   df['outcomes'].str.contains('DEATH', na=False).sum(),
    '생명위협': df['outcomes'].str.contains('LIFE THREATENING', na=False).sum(),
    '입원':   df['outcomes'].str.contains('HOSPITALIZATION', na=False).sum(),
    '응급실':  df['outcomes'].str.contains('EMERGENCY', na=False).sum(),
    '병원방문': df['outcomes'].str.contains('DOCTOR|VISIT', na=False).sum(),
    '비심각':  len(df[df['serious'] == False]),
})
print("필터 전 원값")
print(data_counts.to_string())
print(f"\n'EMERGENCY' 매칭: {data_counts['응급실']}건 → data_counts[data_counts>0] 에서 제외됨")

data_counts = data_counts[data_counts > 0]
percents = 100. * data_counts / data_counts.sum()

fig, ax = plt.subplots(figsize=(9, 6.5))
colors = [plt.cm.Reds(i) for i in np.linspace(0.9, 0.4, len(data_counts))]
ax.pie(data_counts, labels=data_counts.index, autopct='%.1f%%', startangle=90,
       counterclock=False, colors=colors,
       wedgeprops={'width':0.5, 'edgecolor':'w'}, pctdistance=0.75, labeldistance=1.1)
ax.set_title('부작용 분포 결과 [원본 재현]', fontsize=15, fontweight='bold', pad=24)
plt.tight_layout()
plt.savefig(FIG_DIR / 'orig_04_outcome_donut.png', bbox_inches='tight')
plt.show()

print(f"\n분모 = 슬라이스 합계 {data_counts.sum():,}  (전체 행 {len(df):,} 보다 {data_counts.sum()-len(df):,} 많음)")
print(f"비심각 슬라이스        {percents['비심각']:.1f}%")
print(f"나머지 슬라이스 합계   {100-percents['비심각']:.1f}%   ← 보고서의 '43.9%'")

## 9. 연령대 분포 파이 (원본 cell 19)

In [ ]:
age_counts = df['age_cate'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', startangle=140,
       colors=plt.cm.Reds(np.linspace(0.3, 0.7, len(age_counts))),
       wedgeprops={'edgecolor':'white', 'linewidth':2})
ax.set_title('연령대별 분포 비율 [원본 재현]', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'orig_05_age_pie.png', bbox_inches='tight')
plt.show()
print(age_counts.to_string())
print(f"\n분모: 연령 유효 {age_counts.sum():,}행 (전체 {len(df):,}행의 {age_counts.sum()/len(df)*100:.1f}%)")

## 10. 스택바의 연령 구간 (원본 cell 13)

원본은 파이 차트와 **다른 방식**으로 연령을 구간화한다. 두 결과를 비교한다.

In [ ]:
bins = [0, 6, 20, 60, 120]
labels = ['영유아(0-5)', '청소년(6-19)', '성인(20-59)', '노인(66+)']
cut_counts = pd.cut(df['Age_Year'], bins=bins, labels=labels).value_counts()

print(f"{'구간':14s} {'cell10 함수':>12} {'cell13 pd.cut':>14} {'차이':>8}")
for fn_lab, cut_lab in zip(['영유아(0-5)','청소년(6-19)','성인(20-59)','노인(60+)'], labels):
    a = age_counts.get(fn_lab, 0); b = cut_counts.get(cut_lab, 0)
    print(f"{fn_lab:14s} {a:>12,} {b:>14,} {b-a:>+8,}")
print(f"{'합계':14s} {age_counts.sum():>12,} {cut_counts.sum():>14,} {cut_counts.sum()-age_counts.sum():>+8,}")
print(f"\n0세 행 수: {(df.Age_Year == 0).sum():,}  ← pd.cut(right=True) 이므로 (0,6] 구간에서 제외됨")
print("라벨 '노인(66+)' 은 실제 구간 (60,120] 과 불일치 (오표기)")